# Holo4-35B-A3B on Kaggle: OpenAI-compatible API + ngrok

Serves H Company's Holo4-35B-A3B (35B MoE, 3B active, vision + tool calling) as a 4-bit GGUF
through `llama.cpp`'s `llama-server`. It exposes OpenAI-style endpoints
(`/v1/chat/completions`, `/v1/completions`, `/v1/models`, `/v1/embeddings`) and tunnels them out with ngrok.

**Setup before running**
1. Session options → Accelerator: **GPU T4 x2** (about 30 GB VRAM total). A single 16 GB GPU needs `N_CPU_MOE` raised in cell 5.
2. Session options → Internet: **On**.
3. Add-ons → Secrets: `NGROK_AUTHTOKEN`, `NGROK_DOMAIN` (e.g. `my-name.ngrok-free.app`, no `https://`).
   Optional: `HF_TOKEN` (faster downloads), `API_KEY` (protects your endpoint with a bearer key).

**Why GGUF / llama.cpp?** T4 and P100 GPUs have no bf16/FP8/FP4 support, so FP8 and NVFP4 checkpoints
are not usable. Q4_K_M is about 20 GB, which fits across 2x T4 with room for KV cache and the vision projector.

## 1. Check GPUs

In [1]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

index, name, memory.total [MiB], compute_cap
0, Tesla T4, 15360 MiB, 7.5
1, Tesla T4, 15360 MiB, 7.5


## 2. Install Python deps

In [2]:
!pip install -q pyngrok huggingface_hub openai pillow requests

## 3. Build llama.cpp with CUDA (about 10 min the first time; cached under /kaggle/working/binaries)

In [3]:
import subprocess, os, shutil, textwrap

BUILD_SH = r'''
set -o pipefail
export PATH=/usr/local/cuda/bin:$PATH
export CUDACXX=$(which nvcc || echo /usr/local/cuda/bin/nvcc)
cd /kaggle/working
BUILDDIR=/kaggle/working/binaries/llama.cpp
if [ -x "$BUILDDIR/build/bin/llama-server" ] && [ -f "$BUILDDIR/.build-complete" ]; then echo "using cached llama.cpp build in $BUILDDIR"; exit 0; fi
mkdir -p /kaggle/working/binaries

echo "== internet check =="; git ls-remote https://github.com/ggml-org/llama.cpp HEAD || { echo "NO INTERNET: enable Internet in session options"; exit 11; }
echo "== nvcc =="; $CUDACXX --version || { echo "nvcc not found"; exit 12; }

CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -1 | tr -d '. \r')
CC=${CC:-75}
echo "CUDA arch: $CC"

# CUDA 13 on Kaggle has no libcuda.so link stub, so CMake can't create CUDA::cuda_driver.
# Point it at a stub from the toolkit if present, else at the real driver lib (libcuda.so.1).
STUBDIR=/kaggle/working/cudastub
mkdir -p $STUBDIR
LIBCUDA=$(find /usr/local/cuda* -path '*stubs*' -name 'libcuda.so' 2>/dev/null | head -1)
[ -z "$LIBCUDA" ] && LIBCUDA=$(find /usr/lib /usr/lib64 /lib /usr/local/nvidia -name 'libcuda.so.1' 2>/dev/null | head -1)
echo "libcuda used for linking: $LIBCUDA"
[ -z "$LIBCUDA" ] && { echo "libcuda not found anywhere"; exit 16; }
ln -sf "$LIBCUDA" $STUBDIR/libcuda.so

rm -rf "$BUILDDIR"
git clone --depth 1 https://github.com/ggml-org/llama.cpp "$BUILDDIR" || exit 13
cd "$BUILDDIR"
cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=$CC -DCMAKE_CUDA_COMPILER=$CUDACXX \
      -DCMAKE_LIBRARY_PATH=$STUBDIR -DCUDA_DRIVER_LIBRARY=$STUBDIR/libcuda.so \
      -DCMAKE_EXE_LINKER_FLAGS="-L$STUBDIR -Wl,-rpath-link,$STUBDIR" \
      -DCMAKE_BUILD_TYPE=Release -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF 2>&1 | tail -40 || exit 14
cmake --build build --config Release -j $(nproc) --target llama-server 2>&1 | tail -60 || exit 15
touch "$BUILDDIR/.build-complete"
ls -la "$BUILDDIR/build/bin/llama-server"
'''
open("/kaggle/working/build.sh", "w").write(BUILD_SH)
r = subprocess.run(["bash", "/kaggle/working/build.sh"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
print("exit code:", r.returncode)
if r.returncode != 0 or not os.path.exists("/kaggle/working/binaries/llama.cpp/build/bin/llama-server") or not os.path.exists("/kaggle/working/binaries/llama.cpp/.build-complete"):
    raise RuntimeError(
        f"llama.cpp build failed (exit code {r.returncode})\n"
        f"--- STDOUT (tail) ---\n{r.stdout[-2500:]}\n"
        f"--- STDERR (tail) ---\n{r.stderr[-1500:]}"
    )

cuda/targets/x86_64-linux/include/cccl (found version "13.0.88")
-- CUDA Toolkit found
-- The CUDA compiler identification is NVIDIA 13.0.88 with host compiler GNU 13.3.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Using CMAKE_CUDA_ARCHITECTURES=75 CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- FlashAttention K-V type combinations: f16-f16;q4_0-q4_0;q8_0-q8_0;bf16-bf16;f8_e4m3-f8_e4m3
-- Found NCCL: /usr/lib/x86_64-linux-gnu/libnccl.so
-- CUDA host compiler is GNU 13.3.0
-- Including CUDA backend
-- ggml version: 0.29.0
-- ggml commit:  f7b886f
-- Found OpenSSL: /usr/lib/x86_64-linux-gnu/libcrypto.so (found version "3.0.13")
-- Performing Test OPENSSL_VERSION_SUPPORTED
-- Performing Test OPENSSL_VERSION_SUPPORTED - Success
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: llama-app
--

## 4. Load secrets and download the quantized model

In [4]:
import os, shutil
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_secret(name, required=False):
    try:
        return secrets.get_secret(name)
    except Exception:
        if required:
            raise RuntimeError(f"Missing Kaggle secret: {name}")
        return None

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", required=True)
NGROK_DOMAIN = get_secret("NGROK_DOMAIN", required=True).replace("https://", "").replace("http://", "").strip("/ ")
HF_TOKEN = get_secret("HF_TOKEN")
API_KEY = get_secret("API_KEY")  # optional; if set, clients must send "Authorization: Bearer <API_KEY>"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# ---- Model config ----
REPO = "Hcompany/Holo4-35B-A3B-GGUF"   # official GGUF repo
QUANT = "Q4_K_M"                       # ~21 GB. Use "Q4_0" (~18 GB) if you need more KV headroom
MODEL_ALIAS = "holo4-35b-a3b"          # model name clients will use

from huggingface_hub import list_repo_files, hf_hub_download

files = list_repo_files(REPO)
print("\n".join(f for f in files if f.endswith(".gguf")))

weights = sorted(f for f in files if f.endswith(".gguf")
                 and QUANT.lower() in f.lower() and "mmproj" not in f.lower())
if not weights:
    raise RuntimeError(f"No {QUANT} GGUF found in {REPO}; pick a quant from the list above.")

# /kaggle/working is capped at ~20 GB, but the model is ~21 GB. Use temp storage instead.
shutil.rmtree("/kaggle/working/models", ignore_errors=True)   # drop the partial download, if any
MODEL_DIR = "/kaggle/temp/models" if os.path.isdir("/kaggle/temp") else "/tmp/models"
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Downloading to {MODEL_DIR}, free space: {shutil.disk_usage(MODEL_DIR).free / 1e9:.1f} GB")

local_weights = [hf_hub_download(REPO, f, local_dir=MODEL_DIR) for f in weights]
MODEL_PATH = local_weights[0]   # for sharded files llama.cpp finds the rest from the first shard

mmprojs = sorted(f for f in files if "mmproj" in f.lower() and f.endswith(".gguf"))
MMPROJ_PATH = None
if mmprojs:
    pick = next((f for f in mmprojs if "f16" in f.lower()), mmprojs[0])
    MMPROJ_PATH = hf_hub_download(REPO, pick, local_dir=MODEL_DIR)
else:
    print("WARNING: no mmproj file in repo, vision input will be disabled.")

print("Model :", MODEL_PATH)
print("mmproj:", MMPROJ_PATH)

mmproj.f16.gguf
q4_k_m.gguf
Model : /tmp/models/q4_k_m.gguf
mmproj: /tmp/models/mmproj.f16.gguf


## 5. Start the OpenAI-compatible server

In [5]:
import subprocess, time, requests, shlex

PORT = 8000
CTX = 32768          # total context; split across parallel slots
PARALLEL = 2         # concurrent requests (each gets CTX/PARALLEL tokens)
KV_QUANT = "q8_0"    # KV cache quantization (q8_0 halves KV memory; use "f16" if you see problems)
N_CPU_MOE = 0        # keep the experts of N layers in system RAM. 0 on 2x T4. Try 10-20 on a single 16 GB GPU

cmd = [
    "/kaggle/working/binaries/llama.cpp/build/bin/llama-server",
    "-m", MODEL_PATH,
    "--alias", MODEL_ALIAS,
    "--host", "0.0.0.0", "--port", str(PORT),
    "-ngl", "999",
    "--split-mode", "layer",
    "-c", str(CTX),
    "-np", str(PARALLEL),
    "-ctk", KV_QUANT, "-ctv", KV_QUANT,
    "--jinja",                       # model's own chat template (tool calling + vision messages)
]
if MMPROJ_PATH:
    cmd += ["--mmproj", MMPROJ_PATH]
if N_CPU_MOE:
    cmd += ["--n-cpu-moe", str(N_CPU_MOE)]
if API_KEY:
    cmd += ["--api-key", API_KEY]

LOG = "/kaggle/working/llama-server.log"
subprocess.run(["pkill", "-f", "llama-server"], check=False)
time.sleep(2)
server = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("Launched:", " ".join(shlex.quote(c) for c in cmd if c != API_KEY))

# wait for the model to load
t0 = time.time()
while True:
    if server.poll() is not None:
        print(open(LOG).read()[-4000:])
        raise RuntimeError("llama-server exited, see log above")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    if time.time() - t0 > 1200:
        print(open(LOG).read()[-4000:])
        raise TimeoutError("Server did not become healthy in 20 min")
    time.sleep(5)
print(f"Server ready after {time.time()-t0:.0f}s")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

Launched: /kaggle/working/binaries/llama.cpp/build/bin/llama-server -m /tmp/models/q4_k_m.gguf --alias holo4-35b-a3b --host 0.0.0.0 --port 8000 -ngl 999 --split-mode layer -c 32768 -np 2 -ctk q8_0 -ctv q8_0 --jinja --mmproj /tmp/models/mmproj.f16.gguf --api-key
Server ready after 100s
index, memory.used [MiB], memory.total [MiB]
0, 11915 MiB, 15360 MiB
1, 10253 MiB, 15360 MiB


## 6. Open the ngrok tunnel on your reserved domain

In [6]:
import subprocess, time, requests, shlex

PORT = 8000
CTX = 32768          # total context; split across parallel slots
PARALLEL = 2         # concurrent requests (each gets CTX/PARALLEL tokens)
KV_QUANT = "q8_0"    # KV cache quantization (q8_0 halves KV memory; use "f16" if you see problems)
N_CPU_MOE = 0        # keep the experts of N layers in system RAM. 0 on 2x T4. Try 10-20 on a single 16 GB GPU

cmd = [
    "/kaggle/working/binaries/llama.cpp/build/bin/llama-server",
    "-m", MODEL_PATH,
    "--alias", MODEL_ALIAS,
    "--host", "0.0.0.0", "--port", str(PORT),
    "-ngl", "999",
    "--split-mode", "layer",
    "-c", str(CTX),
    "-np", str(PARALLEL),
    "-ctk", KV_QUANT, "-ctv", KV_QUANT,
    "--jinja",                       # model's own chat template (tool calling + vision messages)
]
if MMPROJ_PATH:
    cmd += ["--mmproj", MMPROJ_PATH]
if N_CPU_MOE:
    cmd += ["--n-cpu-moe", str(N_CPU_MOE)]
if API_KEY:
    cmd += ["--api-key", API_KEY]

LOG = "/kaggle/working/llama-server.log"
subprocess.run(["pkill", "-f", "llama-server"], check=False)
time.sleep(2)
server = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("Launched:", " ".join(shlex.quote(c) for c in cmd if c != API_KEY))

# wait for the model to load
t0 = time.time()
while True:
    if server.poll() is not None:
        print(open(LOG).read()[-4000:])
        raise RuntimeError("llama-server exited, see log above")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    if time.time() - t0 > 1200:
        print(open(LOG).read()[-4000:])
        raise TimeoutError("Server did not become healthy in 20 min")
    time.sleep(5)
print(f"Server ready after {time.time()-t0:.0f}s")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

Launched: /kaggle/working/binaries/llama.cpp/build/bin/llama-server -m /tmp/models/q4_k_m.gguf --alias holo4-35b-a3b --host 0.0.0.0 --port 8000 -ngl 999 --split-mode layer -c 32768 -np 2 -ctk q8_0 -ctv q8_0 --jinja --mmproj /tmp/models/mmproj.f16.gguf --api-key
Server ready after 10s
index, memory.used [MiB], memory.total [MiB]
0, 11915 MiB, 15360 MiB
1, 10253 MiB, 15360 MiB


## 7. Smoke test (text + vision + streaming) through the public URL

In [11]:
import os
try:
    BASE_URL
except NameError:
    BASE_URL = f"https://{NGROK_DOMAIN}/v1"      # NGROK_DOMAIN comes from the secrets cell
    print("BASE_URL was not set, using", BASE_URL)
import base64, io
from openai import OpenAI
from PIL import Image, ImageDraw

client = OpenAI(base_url=BASE_URL, api_key=API_KEY or "sk-none")

print(client.models.list().data[0].id)

# text
r = client.chat.completions.create(
    model=MODEL_ALIAS,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
    max_tokens=64,
)
print(r.choices[0].message.content)


# vision (draws a button and asks where it is)
img = Image.new("RGB", (640, 360), "white")
d = ImageDraw.Draw(img)
d.rectangle([420, 250, 560, 310], fill="royalblue")
d.text((460, 272), "Submit", fill="white")
buf = io.BytesIO(); img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode()

r = client.chat.completions.create(
    model=MODEL_ALIAS,
    messages=[{"role": "user", "content": [
        {"type": "text", "text": "Describe the UI element in this screenshot and roughly where it is."},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
    ]}],
    max_tokens=200,
)
print(r.choices[0].message.content)

# streaming
for chunk in client.chat.completions.create(
    model=MODEL_ALIAS, stream=True, max_tokens=60,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
):
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

holo4-35b-a3b

The UI element is a blue rectangular button with the label "Submit" in white text. It is located in the bottom right corner of the image.



In [10]:
from pyngrok import ngrok, conf
import subprocess, time, requests

PORT = globals().get("PORT", 8000)

ngrok.kill()
try:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
except Exception as e:
    if "already set" in str(e).lower():
        print("Auth token already configured, continuing.")
    else:
        raise

PUBLIC_URL = f"https://{NGROK_DOMAIN}"
ngrok_cli_proc = None
tunnel = None

# 1) newer ngrok agent: "url" option
try:
    tunnel = ngrok.connect(addr=PORT, proto="http", url=PUBLIC_URL)
    print("Tunnel opened with url=")
except Exception as e1:
    print("url= failed:", str(e1)[:200])
    # 2) older ngrok agent: "domain" option
    try:
        tunnel = ngrok.connect(addr=PORT, proto="http", domain=NGROK_DOMAIN)
        print("Tunnel opened with domain=")
    except Exception as e2:
        print("domain= failed:", str(e2)[:200])
        # 3) run the ngrok binary directly
        ngrok_path = conf.get_default().ngrok_path
        for flag in ("--url", "--domain"):
            cmd = [ngrok_path, "http", str(PORT), f"{flag}={PUBLIC_URL if flag == '--url' else NGROK_DOMAIN}", "--log=stdout"]
            ngrok_cli_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            time.sleep(6)
            if ngrok_cli_proc.poll() is None:
                print(f"Tunnel opened via ngrok CLI ({flag})")
                break
            print(f"CLI with {flag} exited:", ngrok_cli_proc.stdout.read()[-600:])
            ngrok_cli_proc = None
        else:
            raise RuntimeError("Could not open the ngrok tunnel. Check that NGROK_DOMAIN is a domain reserved on your ngrok account.")

BASE_URL = f"{PUBLIC_URL}/v1"

# verify through the public URL
hdrs = {"ngrok-skip-browser-warning": "true"}
if API_KEY:
    hdrs["Authorization"] = f"Bearer {API_KEY}"
r = requests.get(f"{BASE_URL}/models", headers=hdrs, timeout=30)
print("Public check:", r.status_code)
print("Public base URL:", BASE_URL)
print("Model name     :", MODEL_ALIAS)

Tunnel opened with url=                                                                             
Public check: 200
Public base URL: https://cherub-yo-yo-daffodil.ngrok-free.dev/v1
Model name     : holo4-35b-a3b


In [9]:
import subprocess, requests
print(subprocess.run("pgrep -af llama-server | head -3", shell=True, capture_output=True, text=True).stdout or "llama-server NOT running")
try:
    print("local health:", requests.get("http://127.0.0.1:8000/health", timeout=3).status_code)
except Exception as e:
    print("local server not reachable:", str(e)[:100])

13305 /kaggle/working/binaries/llama.cpp/build/bin/llama-server -m /tmp/models/q4_k_m.gguf --alias holo4-35b-a3b --host 0.0.0.0 --port 8000 -ngl 999 --split-mode layer -c 32768 -np 2 -ctk q8_0 -ctv q8_0 --jinja --mmproj /tmp/models/mmproj.f16.gguf --api-key heytherethisismyapikey
13331 /bin/sh -c pgrep -af llama-server | head -3

local health: 200


## 8. Keep the session alive

Leave this cell running; Kaggle stops the server when the notebook session ends (max about 9-12 h per session).
Use the endpoint from anywhere:

```python
from openai import OpenAI
client = OpenAI(base_url="https://YOUR_DOMAIN/v1", api_key="YOUR_API_KEY_OR_ANYTHING")
client.chat.completions.create(model="holo4-35b-a3b", messages=[{"role": "user", "content": "hi"}])
```
```bash
curl https://YOUR_DOMAIN/v1/chat/completions \
  -H "Authorization: Bearer YOUR_API_KEY" -H "Content-Type: application/json" \
  -d '{"model":"holo4-35b-a3b","messages":[{"role":"user","content":"hi"}]}'
```

In [ ]:
import time
print("Serving at", BASE_URL, "(interrupt the cell to stop)")
try:
    while True:
        if server.poll() is not None:
            print("llama-server died. Last log lines:\n", open(LOG).read()[-3000:])
            break
        time.sleep(60)
except KeyboardInterrupt:
    pass
finally:
    ngrok.kill()
    server.terminate()